# AI-Based Network Intrusion Detection System (NIDS)
## ?? Phase 1: Exploratory Data Analysis (EDA) & Data Ingestion

This notebook explores the statistical distributions, flow metrics, and class imbalances in network intrusion telemetry.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_dataset
from src.imbalance_handler import get_class_weights

sns.set_theme(style="whitegrid")
%matplotlib inline

### 1. Ingest Dataset

In [ ]:
df = load_dataset(dataset_name="cic-ids2017", sample_size=50000)
print(f"Loaded dataset shape: {df.shape}")
df.head()

### 2. Inspect Class Distribution & Severe Imbalances

In [ ]:
plt.figure(figsize=(10, 4))
order = df['Label'].value_counts().index
sns.countplot(data=df, y='Label', order=order, palette='viridis')
plt.title('Attack Category Distribution (Flow Counts)')
plt.xlabel('Count')
plt.ylabel('Threat Class')
plt.show()

### 3. Flow Durations & Rates by Threat Category

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df['Log_Flow_Duration'] = np.log10(np.maximum(df['Flow Duration'], 1.0))
sns.boxplot(data=df, x='Label', y='Log_Flow_Duration', ax=ax1, palette='Set2')
ax1.set_title('Log Flow Duration by Attack')
ax1.tick_params(axis='x', rotation=30)

df['Log_Flow_Bytes_s'] = np.log10(np.maximum(df['Flow Bytes/s'], 1.0))
sns.boxplot(data=df, x='Label', y='Log_Flow_Bytes_s', ax=ax2, palette='Set3')
ax2.set_title('Log Flow Bytes/sec by Attack')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 4. Computed Class Weights for Mitigation

In [ ]:
weights = get_class_weights(df['Label'])
pd.DataFrame(list(weights.items()), columns=['Threat Class', 'Loss Function Weight']).sort_values('Loss Function Weight', ascending=False)